## מנוסחה לקוד: תהליך התרגום ושגיאות נפוצות

לתרגם משוואה פיזיקלית לקוד זו לא רק "העתקת סימנים" — יש לזה תהליך שיטתי, שאפשר וכדאי להפעיל בכל פעם מחדש. בסעיף הזה נעבור על התהליך הזה צעד-צעד על דוגמה מלאה, ואז נבנה יחד "מאגר באגים": שלושת סוגי הטעויות הנפוצים ביותר כשהתהליך משתבש. באגים מהסוג הזה מסוכנים במיוחד, כי הקוד **רץ בלי שגיאה** — הוא פשוט מחזיר תוצאה פיזיקלית לא נכונה, בלי שום התראה.

### תהליך שיטתי: מנוסחה לביטוי ב-NumPy

בכל פעם שמתרגמים משוואה פיזיקלית לקוד, שווה לעבור על ארבעה שלבים, בסדר הזה:

1. **זיהוי כל גורם במשוואה** — האם הוא סקלר קבוע (מספר בודד, כמו מהירות התחלתית אחת), מערך (למשל טווח זמנים, או תוצאה של כמה ריצות), או קבוע פיזיקלי (‎`scipy.constants`‎)?
2. **החלטה על סדר הפעולות והביטוי המתאים ב-NumPy** — אילו סוגריים נדרשים כדי שסדר החישוב יתאים לנוסחה? אילו פעולות פועלות איבר-איבר (סעיף 7.7), ואילו הן סקלר מול מערך?
3. **כתיבת הביטוי שורה-שורה** — לא לנסות לדחוס את כל הנוסחה לשורה אחת מסובכת בניסיון הראשון. לבנות בהדרגה, ולהדפיס תוצאות ביניים כדי לוודא שכל שלב עשה מה שהתכוונתם.
4. **בדיקת סבירות בסוף** — יחידות (כל הגדלים עדיין ב-SI?), סדר גודל (התוצאה הגיונית פיזיקלית?), ומקרי קצה (מה קורה בערכי קיצון, כמו זווית של $0°$ או $90°$?).

### דוגמה: מהנוסחה לקוד — טווח הזריקה

נתרגם את נוסחת הטווח האופקי של זריקה:

$$R(v_0,\theta) = \frac{v_0^2 \sin(2\theta)}{g}$$

**שלב 1 — זיהוי גורמים:** $v_0$ ו-$\theta$ הם סקלרים (מהירות שיגור וזווית שיגור בודדות), $g$ הוא קבוע פיזיקלי. הזווית $\theta$ נתונה במעלות — ולכן, כמו בכל שימוש בפונקציה טריגונומטרית של NumPy (סעיף 7.7), חייבים להמיר אותה לרדיאנים **לפני** שמכניסים אותה ל-‎`np.sin`‎.

**שלב 2 — סדר הפעולות:** קודם ‎`2*theta`‎ (בתוך ה-‎`sin`‎), אז ‎`np.sin(...)`‎, אז ‎`v0**2 * ...`‎, ולבסוף חילוק ב-‎`g`‎. כל צעד כזה יהפוך לשורת קוד נפרדת.

In [ ]:
import numpy as np
from scipy import constants

g = constants.g    # m/s^2
v0 = 20.0           # m/s
theta_deg = 45      # מעלות

# שלב 3: כתיבה שורה-שורה
theta = np.radians(theta_deg)          # המרה לרדיאנים לפני sin
numerator = v0**2 * np.sin(2 * theta)
R = numerator / g

print(f"R = {R:.2f} m")

# שלב 4: בדיקת סבירות - מקרי קצה
print("R בזווית 0°:", round(v0**2 * np.sin(2 * np.radians(0)) / g, 6))    # ציפייה: 0
print("R בזווית 90°:", round(v0**2 * np.sin(2 * np.radians(90)) / g, 6))  # ציפייה: 0

התוצאה, כ-40.8 מטר עבור $v_0=20$ מ/ש בזווית $45°$, נמצאת בסדר גודל סביר לזריקת כדור ביד. בדיקת מקרי הקצה מאששת את הפיזיקה: בזווית $0°$ הכדור לא עף אופקית בכלל (הטווח 0), ובזווית $90°$ הוא עולה ויורד באותה נקודה (הטווח גם כן 0) — בדיוק כפי שמצפים ממשוואת הטווח.

### מנוסחה לפונקציה: חתימה ודוגמה

כשנוסחה עובדת, כדאי לעטוף אותה בפונקציה, כדי להשתמש בה שוב עם מספרים אחרים. שני דברים מגדירים פונקציה כזו עוד לפני שכותבים אותה:

- **חתימה**: שם הפונקציה והפרמטרים, עם היחידות של כל אחד. למשל ‎`launch_range(v0, theta_deg, g)`‎.
- **דוגמה**: קריאה אחת שאת התוצאה שלה יודעים מראש. עם $g = 10$, $v_0 = 20$ ו-$\theta = 45°$ הנוסחה נותנת $20^2 \cdot \sin 90° / 10 = 40$ מטר.

כותבים את הפונקציה, ומיד מריצים את הדוגמה: אם יוצא 40.0, הפונקציה עושה מה שהחתימה מבטיחה. מכאן אפשר לבנות עליה פונקציה נוספת **בלי לכתוב את הנוסחה שוב**: ‎`lands_beyond`‎ משתמשת ב-‎`launch_range`‎ כדי לבדוק אם הכדור נוחת רחוק יותר ממרחק נתון.

In [ ]:
def launch_range(v0, theta_deg, g):
    """Range (m); v0 in m/s, theta_deg in degrees, g in m/s^2."""
    theta = np.radians(theta_deg)
    return v0**2 * np.sin(2 * theta) / g

print(launch_range(20.0, 45.0, 10.0))   # the example: 40.0


def lands_beyond(v0, theta_deg, g, distance):
    """True if the ball lands farther than distance (m)."""
    return launch_range(v0, theta_deg, g) > distance

print(lands_beyond(20.0, 45.0, 10.0, 30.0))   # 40 > 30, so True

### בדקו את עצמכם

```python
v0, g = 20.0, 10.0
print(v0**2 / 2 * g)
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מה תדפיס שורת ההדפסה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "2000.0",
                "correct": true,
                "feedback": "נכון: חילוק וכפל מחושבים משמאל לימין: קודם 400 חלקי 2, ואז כפול 10. כדי לחלק ב-2g צריך סוגריים."
            },
            {
                "answer": "20.0",
                "correct": false,
                "feedback": "לא: זה הגובה המרבי הנכון, אבל בלי סוגריים סביב 2 כפול g מקבלים משהו אחר."
            },
            {
                "answer": "200.0",
                "correct": false,
                "feedback": "לא: אחרי החילוק ב-2 עוד כופלים ב-10."
            },
            {
                "answer": "שגיאה",
                "correct": false,
                "feedback": "לא: הביטוי חוקי, וזו בדיוק הסכנה."
            }
        ]
    },
    {
        "question": "בתרגום נוסחה עם סינוס של זווית שנתונה במעלות, מתי ממירים את הזווית לרדיאנים?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "לפני שמכניסים אותה לפונקציית הסינוס של NumPy, כי היא מצפה לרדיאנים",
                "correct": true,
                "feedback": "נכון: קלט במעלות מחושב כאילו הוא ברדיאנים, וזו זווית אחרת לגמרי."
            },
            {
                "answer": "בסוף, על התוצאה",
                "correct": false,
                "feedback": "לא: אחרי שהסינוס חושב על זווית שגויה, כבר אי אפשר לתקן."
            },
            {
                "answer": "לא צריך: NumPy מזהה שהזווית במעלות",
                "correct": false,
                "feedback": "לא: NumPy לא יודע באיזו יחידה הזווית; הוא תמיד מניח רדיאנים."
            },
            {
                "answer": "רק כשהזווית גדולה מ-90 מעלות",
                "correct": false,
                "feedback": "לא: כל זווית במעלות צריכה המרה."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### נסו בעצמכם

הפעילו את אותו תהליך על נוסחת **הגובה המרבי** של הזריקה:

$$y_{max}(v_0,\theta) = \frac{(v_0\sin\theta)^2}{2g}$$

השלימו את הפונקציה ‎`max_height(v0, theta_deg, g)`‎ לפי החתימה, ובדקו אותה מול הדוגמה שבתא: בזריקה אנכית ($\theta = 90°$) עם $v_0 = 20$ ו-$g = 10$, הגובה הוא $20^2/20 = 20$ מטר. שימו לב: הריבוע חל על **התוצאה** של $v_0\sin\theta$, לא על $v_0$ בלבד.

In [ ]:
def max_height(v0, theta_deg, g):
    """Maximum height (m); v0 in m/s, theta_deg in degrees, g in m/s^2."""
    # complete here
    pass

# expected: 20.0
print(max_height(20.0, 90.0, 10.0))

`````{admonition} פתרון
:class: dropdown, tip
```python
def max_height(v0, theta_deg, g):
    """Maximum height (m); v0 in m/s, theta_deg in degrees, g in m/s^2."""
    theta = np.radians(theta_deg)              # convert before np.sin
    return (v0 * np.sin(theta))**2 / (2 * g)   # square of v0*sin(theta)

print(max_height(20.0, 90.0, 10.0))   # 20.0
```
`````

עכשיו, כשראינו איך התהליך אמור להיראות כשהוא עובד נכון — נעבור למאגר הבאגים: מה קורה, בשקט, כשכל אחד מהשלבים האלה משתבש.

### באג 1: קדימות פעולות

כדור שמשחררים ממנוחה מגובה $h$ פוגע בקרקע אחרי $t = \sqrt{2h/g}$ שניות. הקוד הבא מחשב זאת עבור $h = 20$ מטר ו-$g = 10$, כלומר הציפייה היא $\sqrt{4} = 2$ שניות. אבל שורה 3 מדפיסה ‎`0.6324555320336759`‎. באיזו שורה הבאג? מצאו אותו לפני שתפתחו את הפתרון.

In [ ]:
h, g = 20.0, 10.0                   # m, m/s^2
t_fall_buggy = np.sqrt(2 * h) / g
print(t_fall_buggy)

`````{admonition} פתרון
:class: dropdown, tip
הבאג בשורה 2: ‎`np.sqrt(2 * h) / g`‎ מחשב $\sqrt{2h}/g$, לא $\sqrt{2h/g}$. החלוקה ב-‎`g`‎ צריכה להיות **בתוך** השורש:

```python
t_fall_fixed = np.sqrt(2 * h / g)
print(t_fall_fixed)   # 2.0
```
`````

### באג 2: חילוק שלמים

רוצים לחשב את מהירות הממוצע של כדור שעבר 7 מטרים ב-2 שניות, $v = d/t$, כלומר 3.5 מ'/ש'. אבל שורה 4 מדפיסה 3. באיזו שורה הבאג?

In [ ]:
d = 7   # m
t = 2   # s
v_buggy = d // t
print(v_buggy)

`````{admonition} פתרון
:class: dropdown, tip
הבאג בשורה 3: ‎`//`‎ הוא חילוק שלמים (מחזיר את החלק השלם, מעוגל כלפי מטה) — ‎`7 // 2`‎ נותן ‎`3`‎, לא ‎`3.5`‎. עבור גדלים פיזיקליים רציפים צריך את אופרטור החילוק הרגיל ‎`/`‎:

```python
v_fixed = d / t
print(v_fixed)   # 3.5
```

חילוק שלמים שגוי מסוכן במיוחד כי הוא לא זורק שגיאה, ולעיתים אף נותן תוצאה "סבירה למראה" (כמו 3, לא רחוק מ-3.5).
`````

### באג 3: יחידות

מודדים זווית שיגור של $30°$ ורוצים את רכיב המהירות האנכי, $v_y = v_0 \sin\theta$, כלומר $20 \cdot 0.5 = 10$ מ'/ש'. אבל שורה 4 מדפיסה מספר שלילי, ‎`-19.760632481857236`‎. באיזו שורה הבאג?

In [ ]:
v0 = 20.0         # m/s
theta_deg = 30    # degrees
vy_buggy = v0 * np.sin(theta_deg)
print(vy_buggy)

`````{admonition} פתרון
:class: dropdown, tip
הבאג בשורה 3: ‎`np.sin`‎ מצפה לזווית ברדיאנים, ו-‎`theta_deg`‎ הוא במעלות. חייבים להמיר קודם עם ‎`np.radians`‎:

```python
vy_fixed = v0 * np.sin(np.radians(theta_deg))
print(round(vy_fixed, 2))   # 10.0
```

בלי ‎`round`‎ השורה מדפיסה ‎`9.999999999999998`‎: מספרים עשרוניים במחשב מדויקים רק עד כ-16 ספרות.
`````

### נסו בעצמכם

הקוד הבא מחשב את האנרגיה הפוטנציאלית הממוצעת של שלושה גופים, $\bar{E} = mg\bar{h}$, כאשר ‎`heights_cm`‎ הוא מערך גבהים בסנטימטרים. גובה ממוצע של כ-1.4 מטר וגוף של 2 ק"ג נותנים כ-28 ג'אול, אבל שורה 4 מדפיסה ‎`2811.239666666667`‎. איזה מסוגי הבאגים שראינו יש כאן, ובאיזו שורה? תקנו אותו.

In [ ]:
m = 2.0                                 # kg
heights_cm = np.array([150, 200, 80])   # cm
avg_energy_buggy = m * constants.g * heights_cm.mean()
print(avg_energy_buggy)

`````{admonition} פתרון
:class: dropdown, tip
באג יחידות בשורה 3: הגבהים בסנטימטרים, ו-‎`constants.g`‎ במטרים לשנייה בריבוע, ולכן התוצאה גדולה פי 100. ממירים למטרים לפני החישוב:

```python
heights_m = heights_cm / 100                       # cm to m
avg_energy_fixed = m * constants.g * heights_m.mean()
print(avg_energy_fixed)   # 28.112396666666665
```
`````

### בדקו את עצמכם

```python
h, g = 20.0, 10.0
print(9 // 2)               # print 1
print(np.sqrt(2 * h / g))   # print 2
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מה תדפיס שורת ההדפסה הראשונה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "4",
                "correct": true,
                "feedback": "נכון: חילוק שלמים מעגל כלפי מטה, ל-4."
            },
            {
                "answer": "4.5",
                "correct": false,
                "feedback": "לא: זו התוצאה של חילוק רגיל, לא של חילוק שלמים."
            },
            {
                "answer": "5",
                "correct": false,
                "feedback": "לא: חילוק שלמים מעגל כלפי מטה, לא כלפי מעלה."
            },
            {
                "answer": "שגיאה",
                "correct": false,
                "feedback": "לא: זה ביטוי חוקי לגמרי."
            }
        ]
    },
    {
        "question": "מה תדפיס שורת ההדפסה השנייה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "2.0",
                "correct": true,
                "feedback": "נכון: בתוך השורש יש 2 כפול 20 חלקי 10, כלומר 4, והשורש של 4 הוא 2."
            },
            {
                "answer": "בערך 0.632",
                "correct": false,
                "feedback": "לא: זו התוצאה של הגרסה עם הבאג, שבה החלוקה ב-g נמצאת מחוץ לשורש."
            },
            {
                "answer": "4.0",
                "correct": false,
                "feedback": "לא: 4 הוא מה שבתוך השורש; השורש שלו הוא 2."
            },
            {
                "answer": "שגיאה",
                "correct": false,
                "feedback": "לא: הקוד חוקי."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)